# Генерація тестових даних
Ноутбук для наповнення БД синтетичними даними, згенерованими за допомогою LLM.

## 1. Імпорти та конфігурація

In [48]:
import os, json, sys
os.chdir(r"E:/projects/University/BKR/bcr-app/packages/backend")
sys.path.insert(0, '..')

## 2. Підключення до БД

In [49]:
import nest_asyncio
import asyncio
from src.db.session import get_async_sessionmaker

In [50]:
nest_asyncio.apply()

session_factory = get_async_sessionmaker()

In [51]:
from sqlalchemy import delete
from src.db.models import Entry, Goal, Report, User

async def clear_db():
    async with session_factory() as session:
        await session.execute(delete(Entry))
        await session.execute(delete(Report))
        await session.execute(delete(Goal))
        await session.execute(delete(User))
        await session.commit()
        print("DB cleared!")

asyncio.run(clear_db())

2026-03-23 09:17:38,655 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-23 09:17:38,657 INFO sqlalchemy.engine.Engine DELETE FROM entry
2026-03-23 09:17:38,659 INFO sqlalchemy.engine.Engine [cached since 1373s ago] {}
2026-03-23 09:17:38,691 INFO sqlalchemy.engine.Engine DELETE FROM report
2026-03-23 09:17:38,693 INFO sqlalchemy.engine.Engine [cached since 1373s ago] {}
2026-03-23 09:17:38,697 INFO sqlalchemy.engine.Engine DELETE FROM goal
2026-03-23 09:17:38,701 INFO sqlalchemy.engine.Engine [cached since 1373s ago] {}
2026-03-23 09:17:38,708 INFO sqlalchemy.engine.Engine DELETE FROM "user"
2026-03-23 09:17:38,710 INFO sqlalchemy.engine.Engine [cached since 1373s ago] {}
2026-03-23 09:17:38,713 INFO sqlalchemy.engine.Engine COMMIT
DB cleared!


## 3. Генерація даних через LLM

In [52]:
from pydantic import BaseModel
from enum import Enum
from datetime import date
from langchain_core.messages import SystemMessage

In [53]:
class Status(Enum):
    ACTIVE = "active"
    FINISHED = "finished"
    POSTPONE = "postpone"

In [54]:
# --- Моделі для генерації структури (без нотаток) ---

class GoalSeedNoNotes(BaseModel):
    title: str
    description: str | None
    status: Status
    deadline: date | None
    created_at: date
    activity_end: date  # остання дата активності по цілі

class UserSeedNoNotes(BaseModel):
    name: str
    email: str
    language: str
    goals: list[GoalSeedNoNotes]

class SeedStructure(BaseModel):
    users: list[UserSeedNoNotes]


# --- Моделі для фінального результату ---

class Note(BaseModel):
    date_note: date
    note: str
    productivity_score: int

class GoalSeed(BaseModel):
    title: str
    description: str | None
    status: Status
    deadline: date | None
    created_at: date
    notes: list[Note]

class UserSeed(BaseModel):
    name: str
    email: str
    language: str
    goals: list[GoalSeed]

class SeedData(BaseModel):
    users: list[UserSeed]

class NotesOnly(BaseModel):
    notes: list[Note]

In [55]:
from functools import lru_cache
from langchain_openai import ChatOpenAI
from src.core.settings import get_settings

@lru_cache(maxsize=1)
def get_llm() -> ChatOpenAI:
    settings = get_settings()
    return ChatOpenAI(
        model=settings.llm_model,
        api_key=settings.open_ai_key,
    )

llm = get_llm()

### Крок 1 — Структура користувачів і цілей

In [56]:
STRUCTURE_PROMPT = """Generate user and goal structure for a goal-tracking app. NO entries needed.

### Users
3 distinct users with different personalities, lifestyles and goal types.
- User 1: Ukrainian
- User 2: English
- User 3: Ukrainian

### Goals
- User 1: 2-3 goals, activity spans 2-3 months
- User 2: 2-3 goals, activity spans 2-3 months
- User 3: 2-3 goals, activity spans 12 months (set created_at in early 2025)

Goal variety across all users:
- Types: sport, learning, career, health, creative hobby
- Statuses must include:
  - At least one `finished` goal
  - At least one `postpone` goal
  - Rest are `active`

For each goal set realistic created_at and activity_end dates.
Respond with valid JSON only.
"""

In [57]:
structure_llm = llm.with_structured_output(SeedStructure)

structure: SeedStructure = await asyncio.to_thread(
    structure_llm.invoke,
    [SystemMessage(content=STRUCTURE_PROMPT)]
)

for user in structure.users:
    print(f"\n{user.name} ({user.language}):")
    for goal in user.goals:
        print(f"  [{goal.status.value}] {goal.title}: {goal.created_at} → {goal.activity_end}")


Олена Мельник (uk):
  [finished] Півмарафон без зупинок: 2025-03-10 → 2025-06-28
  [active] Англійська для роботи до рівня B2: 2025-04-01 → 2025-08-01
  [active] Стабільний сон і режим: 2025-05-05 → 2025-07-05

Emily Carter (en):
  [active] Complete a 12-week strength training plan: 2025-06-01 → 2025-08-31
  [finished] Finish a UX portfolio case study: 2025-05-20 → 2025-07-12
  [postpone] Daily reading habit: 2025-04-15 → 2025-07-15

Іван Петренко (uk):
  [active] Запустити персональний блог про подорожі: 2025-01-18 → 2026-01-18
  [active] Накопичити фінансову подушку: 2025-02-05 → 2026-02-05
  [postpone] Повернутися до регулярних тренувань: 2025-03-02 → 2026-03-02


e:\projects\University\BKR\bcr-app\packages\backend\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=SeedStructure(users=[User...me.date(2026, 3, 2))])]), input_type=SeedStructure])
  return self.__pydantic_serializer__.to_python(


### Крок 2 — Генерація записів по кожній цілі

In [58]:
ENTRIES_PROMPT = """Generate journal entries for one goal in a personal tracking app.

User: {name}
Language: {language}
Goal: {title}
Description: {description}
Status: {status}
Period: {start} to {end} ({months} months)

REQUIREMENTS:
- Generate EXACTLY {count} entries
- Spread evenly: ~15 entries per month
- Progress arc: early motivation → struggles → adaptation → (resolution if finished/postpone)
- productivity_score varies naturally 1-5, not always high
- Notes: specific, personal, first person — mention real actions, emotions, obstacles
- Date gaps are normal, not every day

Respond with valid JSON only.
"""

notes_llm = llm.with_structured_output(NotesOnly)

async def generate_notes(user: UserSeedNoNotes, goal: GoalSeedNoNotes) -> list[Note]:
    months = max(1, (goal.activity_end.year - goal.created_at.year) * 12
                 + goal.activity_end.month - goal.created_at.month + 1)
    count = months * 15

    prompt = ENTRIES_PROMPT.format(
        name=user.name,
        language=user.language,
        title=goal.title,
        description=goal.description,
        status=goal.status.value,
        start=goal.created_at,
        end=goal.activity_end,
        months=months,
        count=count,
    )
    result = await asyncio.to_thread(
        notes_llm.invoke,
        [SystemMessage(content=prompt)]
    )
    return result.notes

In [59]:
full_users = []

for user in structure.users:
    goals_with_notes = []
    for goal in user.goals:
        print(f"Generating: {user.name} — {goal.title}...")
        notes = await generate_notes(user, goal)
        print(f"  → {len(notes)} entries")

        goals_with_notes.append(GoalSeed(
            title=goal.title,
            description=goal.description,
            status=goal.status,
            deadline=goal.deadline,
            created_at=goal.created_at,
            notes=notes,
        ))

    full_users.append(UserSeed(
        name=user.name,
        email=user.email,
        language=user.language,
        goals=goals_with_notes,
    ))

result = SeedData(users=full_users)

print("\n=== Summary ===")
for user in result.users:
    print(f"\n{user.name}:")
    for goal in user.goals:
        print(f"  [{goal.status.value}] {goal.title}: {len(goal.notes)} entries")

Generating: Олена Мельник — Півмарафон без зупинок...


e:\projects\University\BKR\bcr-app\packages\backend\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=NotesOnly(notes=[Note(dat... productivity_score=4)]), input_type=NotesOnly])
  return self.__pydantic_serializer__.to_python(


  → 58 entries
Generating: Олена Мельник — Англійська для роботи до рівня B2...


e:\projects\University\BKR\bcr-app\packages\backend\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=NotesOnly(notes=[Note(dat... productivity_score=3)]), input_type=NotesOnly])
  return self.__pydantic_serializer__.to_python(


  → 110 entries
Generating: Олена Мельник — Стабільний сон і режим...
  → 49 entries
Generating: Emily Carter — Complete a 12-week strength training plan...


e:\projects\University\BKR\bcr-app\packages\backend\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=NotesOnly(notes=[Note(dat... productivity_score=5)]), input_type=NotesOnly])
  return self.__pydantic_serializer__.to_python(


  → 47 entries
Generating: Emily Carter — Finish a UX portfolio case study...
  → 44 entries
Generating: Emily Carter — Daily reading habit...
  → 48 entries
Generating: Іван Петренко — Запустити персональний блог про подорожі...
  → 218 entries
Generating: Іван Петренко — Накопичити фінансову подушку...
  → 182 entries
Generating: Іван Петренко — Повернутися до регулярних тренувань...
  → 134 entries

=== Summary ===

Олена Мельник:
  [finished] Півмарафон без зупинок: 58 entries
  [active] Англійська для роботи до рівня B2: 110 entries
  [active] Стабільний сон і режим: 49 entries

Emily Carter:
  [active] Complete a 12-week strength training plan: 47 entries
  [finished] Finish a UX portfolio case study: 44 entries
  [postpone] Daily reading habit: 48 entries

Іван Петренко:
  [active] Запустити персональний блог про подорожі: 218 entries
  [active] Накопичити фінансову подушку: 182 entries
  [postpone] Повернутися до регулярних тренувань: 134 entries


In [60]:
from IPython.display import HTML
import json

data = json.dumps(result.model_dump(mode="json"), indent=2, ensure_ascii=False)
display(HTML(f'<pre style="height:500px;overflow:auto">{data}</pre>'))

## 4. Ембединги

In [61]:
from sentence_transformers import SentenceTransformer

In [ ]:
embedding_model = SentenceTransformer('Alibaba-NLP/gte-multilingual-base', trust_remote_code=True)

## 5. Запис у БД

In [ ]:
from src.db.models import Goal, User, Entry


async def seed():
    async with session_factory() as session:
        for user_data in result.users:
            user = User(name=user_data.name, email=user_data.email)
            session.add(user)
            await session.flush()

            for goal_data in user_data.goals:
                goal = Goal(
                    user_id=user.id,
                    title=goal_data.title,
                    description=goal_data.description,
                    status=goal_data.status.value,
                    created_at=goal_data.created_at,
                    deadline=goal_data.deadline,
                )
                session.add(goal)
                await session.flush()

                for entry_data in goal_data.notes:
                    emb = embedding_model.encode(entry_data.note).tolist()

                    entry = Entry(
                        goal_id=goal.id,
                        date_note=entry_data.date_note,
                        note=entry_data.note,
                        productivity_score=entry_data.productivity_score,
                        embedding=emb,
                    )
                    session.add(entry)

        await session.commit()
        print("Done!")

asyncio.run(seed())

## 6. Верифікація

In [64]:
from sqlalchemy import func, select

async def verify():
    async with session_factory() as session:
        for model, label in [(User, 'users'), (Goal, 'goals'), (Entry, 'entries')]:
            count = await session.scalar(select(func.count()).select_from(model))
            print(f"{label}: {count}")

asyncio.run(verify())

2026-03-23 09:23:54,673 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-23 09:23:54,679 INFO sqlalchemy.engine.Engine SELECT count(*) AS count_1 
FROM "user"
2026-03-23 09:23:54,682 INFO sqlalchemy.engine.Engine [generated in 0.00201s] {}
users: 3
2026-03-23 09:23:54,695 INFO sqlalchemy.engine.Engine SELECT count(*) AS count_1 
FROM goal
2026-03-23 09:23:54,695 INFO sqlalchemy.engine.Engine [generated in 0.00115s] {}
goals: 9
2026-03-23 09:23:54,701 INFO sqlalchemy.engine.Engine SELECT count(*) AS count_1 
FROM entry
2026-03-23 09:23:54,704 INFO sqlalchemy.engine.Engine [generated in 0.00114s] {}
entries: 890
2026-03-23 09:23:54,709 INFO sqlalchemy.engine.Engine ROLLBACK


## 7. Re-embedding (запускати якщо entries вже в БД але embedding = NULL)

In [ ]:
from sqlalchemy import select
from src.db.models import Entry

BATCH_SIZE = 64

async def reembed_entries():
    async with session_factory() as session:
        result = await session.execute(
            select(Entry).where(Entry.embedding == None)
        )
        entries = result.scalars().all()
        print(f"Entries without embedding: {len(entries)}")

        for i in range(0, len(entries), BATCH_SIZE):
            batch = entries[i : i + BATCH_SIZE]
            texts = [e.note for e in batch]
            embeddings = embedding_model.encode(texts, batch_size=BATCH_SIZE, show_progress_bar=False)
            for entry, emb in zip(batch, embeddings):
                entry.embedding = emb.tolist()
            await session.commit()
            print(f"  committed {min(i + BATCH_SIZE, len(entries))}/{len(entries)}")

    print("Re-embedding done!")

asyncio.run(reembed_entries())